<a href="https://colab.research.google.com/github/luanavarelaa/puc_aci_final_project/blob/main/ACI_Projeto_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Library Imports

In [ ]:
import os
import gc
import glob
import warnings
import numpy as np
from PIL import Image
from tqdm import tqdm
import kagglehub

# Suppress unnecessary warnings from auxiliary libraries
warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', module='sklearn')

from sklearn.model_selection import StratifiedKFold
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, Flatten, Input, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adadelta

print("TensorFlow Version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

TensorFlow Version: 2.21.0
GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


### 2. LGG MRI Dataset Download

In [ ]:
# 1. Download the LGG dataset via kagglehub
path = kagglehub.dataset_download("mateuszbuda/lgg-mri-segmentation")
print("Dataset path:", path)

Using Colab cache for faster access to the 'lgg-mri-segmentation' dataset.
Dataset path: /kaggle/input/lgg-mri-segmentation


### 3. Image Processing and Labeling

In [ ]:
base_dir = os.path.join(path, "kaggle_3m") if os.path.exists(os.path.join(path, "kaggle_3m")) else path

# 2. Definition of structured output folders by binary class
output_dir = "processed_lgg_dataset"
dir_tumor = os.path.join(output_dir, "tumor")
dir_non_tumor = os.path.join(output_dir, "non_tumor")

os.makedirs(dir_tumor, exist_ok=True)
os.makedirs(dir_non_tumor, exist_ok=True)

# 3. Image mapping (excludes mask files from direct search)
image_paths = glob.glob(f"{base_dir}/*/*[!_mask].tif")
print(f"Total axial slices found: {len(image_paths)}")

# 4. Reading and automatic labeling based on tumor presence in the mask
for img_path in tqdm(image_paths, desc="Labeling and organizing slices"):
    mask_path = img_path.replace(".tif", "_mask.tif")
    if not os.path.exists(mask_path):
        continue

    img = Image.open(img_path)
    mask = Image.open(mask_path)
    mask_arr = np.array(mask)

    # Criterion: if there is any pixel > 0, it is treated as a slice with tumor
    has_tumor = np.max(mask_arr) > 0

    if img.size != (256, 256):
        img = img.resize((256, 256))

    filename = os.path.basename(img_path).replace(".tif", ".png")

    if has_tumor:
        img.save(os.path.join(dir_tumor, filename))
    else:
        img.save(os.path.join(dir_non_tumor, filename))

n_tumor = len(os.listdir(dir_tumor))
n_non_tumor = len(os.listdir(dir_non_tumor))
print(f"\nCompleted! Tumor (Class 1): {n_tumor} | Non-Tumor (Class 0): {n_non_tumor}")

Total axial slices found: 3929


Labeling and organizing slices: 100%|██████████| 3929/3929 [02:01<00:00, 32.45it/s]


Completed! Tumor (Class 1): 1373 | Non-Tumor (Class 0): 2556


### 4. Lightweight Data Loading (Single-Channel RAM Optimization)

In [ ]:
# Loading into memory as single-channel grayscale [0.0, 1.0] (float32) to save RAM
dataset_dir = "processed_lgg_dataset"
classes = {"non_tumor": 0, "tumor": 1}

X_list = []
y_list = []

for class_name, label in classes.items():
    folder_path = os.path.join(dataset_dir, class_name)
    files = glob.glob(os.path.join(folder_path, "*.png"))
    for f in files:
        img = Image.open(f).convert("L")
        img_arr = np.array(img, dtype=np.float32) / 255.0
        X_list.append(img_arr)
        y_list.append(label)

X = np.array(X_list, dtype=np.float32)
y = np.array(y_list, dtype=np.float32)

# Free intermediate memory
del X_list, y_list
gc.collect()

print(f"Total loaded matrices: {X.shape[0]} with dimensions {X.shape[1:]}")

Total loaded matrices: 3929 with dimensions (256, 256)


### 5. PCA Data Augmentation Function Definition

In [ ]:
# Definition of PCA-based augmentation method (90% noise)
def apply_pca_augmentation(images_2d, noise_ratio=0.9):
    augmented_list = []
    t = 1.0 - noise_ratio  # Threshold t = 0.1 for 90% noise

    for img in images_2d:
        flat = img.flatten().reshape(1, -1)
        pca = PCA(n_components=1)
        Z = pca.fit_transform(flat)
        Vr = np.random.uniform(low=t, high=1.0, size=Z.shape)
        Z_prime = Z * Vr  # Hadamard product in latent space
        recon = pca.inverse_transform(Z_prime).reshape(img.shape)
        recon = np.clip(recon, 0.0, 1.0).astype(np.float32)
        augmented_list.append(recon)

    return np.array(augmented_list, dtype=np.float32)

### 6. ResNet50 Architecture Definition (GPU Channel Expansion)

In [ ]:
# ResNet50 definition (channel conversion managed on GPU to save RAM)
def create_resnet50_model(input_shape=(256, 256, 1), transfer_learning=True):
    inputs = Input(shape=input_shape)

    # Replicates the gray channel into 3 channels directly on GPU
    x_3ch = Concatenate()([inputs, inputs, inputs])

    weights = 'imagenet' if transfer_learning else None
    base = ResNet50(weights=weights, include_top=False, input_tensor=x_3ch)

    x = Flatten()(base.output)
    output = Dense(1, activation='sigmoid')(x)

    model = Model(inputs=inputs, outputs=output)
    model.compile(
        optimizer=Adadelta(learning_rate=1.0),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

### 7. Model Training and 10-Fold Cross-Validation

In [ ]:
# Execution of Stratified K-Fold Cross Validation
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
fold_metrics = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):

    print(f"\n==================== Fold {fold + 1}/10 ====================")

    X_train_orig, y_train_orig = X[train_idx], y[train_idx]
    X_test, y_test = X[test_idx], y[test_idx]

    # Applying PCA data augmentation exclusively on the training set
    print("Applying PCA Data Augmentation on training set...")
    X_train_aug = apply_pca_augmentation(X_train_orig, noise_ratio=0.9)
    y_train_aug = y_train_orig.copy()

    # Concatenation and adding the channel dimension (N, 256, 256, 1)
    X_train = np.concatenate([X_train_orig, X_train_aug], axis=0)
    y_train = np.concatenate([y_train_orig, y_train_aug], axis=0)

    X_train = np.expand_dims(X_train, axis=-1)
    X_test_input = np.expand_dims(X_test, axis=-1)

    # Clean RAM
    del X_train_orig, X_train_aug
    gc.collect()

    # Instantiate and train
    model = create_resnet50_model(input_shape=(256, 256, 1), transfer_learning=True)

    model.fit( X_train, y_train, batch_size=10, epochs=50, validation_split=0.1, verbose=1)

    # Evaluation on the fold test set
    y_pred_prob = model.predict(X_test_input)
    y_pred = (y_pred_prob >= 0.5).astype(int).flatten()

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    fold_metrics.append({"acc": acc, "prec": prec, "sen": rec, "spe": spec, "f1": f1})
    print(f"\n[Fold {fold + 1} Results] Accuracy: {acc:.4f} | Sensitivity: {rec:.4f} | Specificity: {spec:.4f} | F1-Score: {f1:.4f}")


==================== Fold 1/10 ====================
Applying PCA Data Augmentation on training set...
Epoch 1/50
 92/637 ━━━━━━━━━━━━━━━━━━━━ 1:03 117ms/step - accuracy: 0.6855 - loss: 4.5142

KeyboardInterrupt: 